# Juiz de Bocha — reconhecedor no Colab

Roda o `Recognizer` do repositório [JonathasmpDEV/juiz-de-bocha-algoritmo](https://github.com/JonathasmpDEV/juiz-de-bocha-algoritmo) (Detectron2 / Mask R-CNN) e a lógica de decisão do juiz (`cloud-run-function/main.py`): acha o **balim** (menor bola) e aponta a **bola vencedora** (mais próxima dele).

**Antes de rodar:**
*Ambiente de execução → Alterar tipo → GPU (T4)*.

**Pesos:** os pesos customizados treinados (`model_final.pth`) não estão no repo nem no Drive, então usamos os pesos COCO do Detectron2 Model Zoo (classe `sports ball`) — o mesmo modelo do `test-recognizer.py`.

## 1. GPU

In [ ]:
!nvidia-smi -L || echo "Sem GPU — vai rodar em CPU (lento)"
import torch, sys
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Python", sys.version.split()[0])

## 2. Instalar Detectron2
Compila a partir do código-fonte (as wheels `cu113/torch1.10` do README não existem para o Python/torch atuais do Colab). Leva ~5–10 min na primeira vez.

In [ ]:
try:
    import detectron2
    print("detectron2 já instalado:", detectron2.__version__)
except ImportError:
    !pip install -q exifread
    !pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
    import detectron2
    print("detectron2 instalado:", detectron2.__version__)

## 3. Clonar o repositório

In [ ]:
import os

REPO_DIR = "/content/juiz-de-bocha-algoritmo"
if not os.path.isdir(REPO_DIR):
    !git clone -q https://github.com/JonathasmpDEV/juiz-de-bocha-algoritmo.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q || true
%cd {REPO_DIR}
!git log --oneline -1

## 4. Carregar o modelo

In [ ]:
import sys, torch
sys.path.insert(0, REPO_DIR)
from recognizer import Recognizer

MODELOS = {
    "R50": ("models/mask_rcnn_R_50_FPN_3x/mask_rcnn_R_50_FPN_3x.yaml",
            "detectron2://COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x/137849600/model_final_f10217.pkl"),
    "X101": ("models/mask_rcnn_X_101_32x8d_FPN_3x/mask_rcnn_X_101_32x8d_FPN_3x.yaml",
             "detectron2://COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x/139653917/model_final_2d9806.pkl"),
}
MODELO = "R50"        # @param ["R50", "X101"]
CONFIANCA = 0.3       # @param {type:"number"}

config_file, weights_file = MODELOS[MODELO]
rec = Recognizer(
    config_file=config_file,
    weights_file=weights_file,
    confidence_threshold=CONFIANCA,
    gpu_id=0 if torch.cuda.is_available() else None,
)
print("Modelo", MODELO, "carregado | classes:", len(rec._class_names))

## 5. Imagem de teste
Por padrão usa `test-cloudrun/test.jpg` do repo. Marque `ENVIAR_IMAGEM` para subir uma foto sua.

In [ ]:
ENVIAR_IMAGEM = False  # @param {type:"boolean"}

caminho = os.path.join(REPO_DIR, "test-cloudrun/test.jpg")
if ENVIAR_IMAGEM:
    from google.colab import files
    enviado = files.upload()
    caminho = os.path.abspath(next(iter(enviado)))

img = Recognizer.read_file(caminho)  # RGB, já com orientação EXIF corrigida
print(caminho, img.shape)

## 6. Detecção

In [ ]:
import time
import matplotlib.pyplot as plt
from detectron2.utils.visualizer import Visualizer

t = time.time()
instancias = rec.predict_data(img)
print(f"Predição: {(time.time() - t) * 1000:.0f} ms | {len(instancias)} instâncias")
for i in instancias:
    print(f"  {i['class']:<12} score={i['score']:.2f} area={i['area']}")

vis = Visualizer(img, rec._metadata).draw_instance_predictions(rec.predict_instances(img))
plt.figure(figsize=(12, 9)); plt.imshow(vis.get_image()); plt.axis("off"); plt.show()

## 7. Decisão do juiz
Mesma lógica de `__process_balls` em `cloud-run-function/main.py`: filtra `sports ball`, separa bolas no meio x nas bordas (margem 10%), pega a menor bola viável do meio como balim (descarta "sujeira" muito menor que as outras), remove sujeira das bordas e ordena as demais pela distância ao balim.

In [ ]:
import math

def distancia(ca, cb):
    return math.hypot(cb["x"] - ca["x"], cb["y"] - ca["y"])

def processar_bolas(img, instancias):
    instancias = [i for i in instancias if i["class"] == "sports ball"]
    bolas, balim, vencedora = [], None, None
    if len(instancias) >= 2:
        margem = 0.10
        x_min, x_max = img.shape[1] * margem, img.shape[1] * (1 - margem)
        y_min, y_max = img.shape[0] * margem, img.shape[0] * (1 - margem)
        no_meio, nas_bordas = [], []
        for inst in instancias:
            c = inst["center"]
            (no_meio if x_min <= c["x"] <= x_max and y_min <= c["y"] <= y_max else nas_bordas).append(inst)
        if no_meio:
            no_meio.sort(key=lambda i: i["area"])
            while True:
                balim = no_meio.pop(0)
                if not no_meio:
                    break
                razao_maior = no_meio[-1]["area"] / balim["area"]
                razao_proxima = no_meio[0]["area"] / balim["area"]
                if math.fabs(razao_maior - razao_proxima) <= 1.5:
                    break  # balim viável
            nas_bordas = [b for b in nas_bordas if b["area"] >= balim["area"] / 2]
            bolas = no_meio + nas_bordas
            for b in bolas:
                b["distance"] = distancia(balim["center"], b["center"])
            bolas.sort(key=lambda b: b["distance"])
            vencedora = bolas[0] if bolas else None
    return bolas, vencedora, balim

bolas, vencedora, balim = processar_bolas(img, instancias)

if balim is None:
    print("Não deu para decidir: menos de 2 bolas detectadas no meio da cancha. Tente baixar CONFIANCA ou usar X101.")
else:
    import cv2
    out = img.copy()
    esp = max(3, int(max(img.shape[:2]) * 0.004))
    def circulo(inst, cor, extra=0):
        b = inst["box"]
        centro = (int(inst["center"]["x"]), int(inst["center"]["y"]))
        eixos = (int((b["x2"] - b["x1"]) / 2) + extra, int((b["y2"] - b["y1"]) / 2) + extra)
        cv2.ellipse(out, centro, eixos, 0, 0, 360, cor, esp)
    for b in bolas[1:]:
        circulo(b, (255, 200, 0))
    circulo(balim, (255, 255, 255))
    if vencedora:
        circulo(vencedora, (50, 230, 50), extra=10)
    print(f"Balim: centro=({balim['center']['x']:.0f}, {balim['center']['y']:.0f}) area={balim['area']}")
    for n, b in enumerate(bolas, 1):
        marca = "  <- VENCEDORA" if b is vencedora else ""
        print(f"  {n}. distância={b['distance']:.0f}px area={b['area']} score={b['score']:.2f}{marca}")
    plt.figure(figsize=(12, 9)); plt.imshow(out); plt.axis("off")
    plt.title("Branco = balim | Verde = vencedora | Amarelo = demais"); plt.show()